In [90]:
!sudo apt-get install zstd
!pip install chess

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 125 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 MB 19.5 MB/s eta 0:00:00:00:0100:01


In [124]:
import subprocess
import io
import chess.pgn

N_GAMES = 50_000
OUTPUT_FILE = "games.txt"

# Download August 2026 Lichess database and stream-decompress it.
wget = subprocess.Popen(
    [
        "wget",
        "-qO-",
        "https://database.lichess.org/standard/lichess_db_standard_rated_2026-08.pgn.zst",
    ],
    stdout=subprocess.PIPE,
)

zstd = subprocess.Popen(
    ["zstd", "-d"],
    stdin=wget.stdout,
    stdout=subprocess.PIPE,
)

# Allow wget to receive SIGPIPE if zstd exits early.
wget.stdout.close()

# Convert bytes → text so python-chess can parse PGN.
pgn_stream = io.TextIOWrapper(
    zstd.stdout,
    encoding="utf-8"
)

games_written = 0

with open(OUTPUT_FILE, "w", encoding="utf-8") as out:

    while games_written < N_GAMES:

        game = chess.pgn.read_game(pgn_stream)

        # End of database
        if game is None:
            break

        # Extract UCI moves
        uci_moves = [
            move.uci()
            for move in game.mainline_moves()
        ]

        # Ignore games with no moves
        if not uci_moves:
            continue

        # One complete game per line
        out.write(" ".join(uci_moves) + "\n")

        games_written += 1

        if games_written % 1000 == 0:
            print(f"Extracted {games_written}/{N_GAMES} games")

pgn_stream.close()

print(f"\nFinished.")
print(f"Games extracted: {games_written}")
print(f"Saved to: {OUTPUT_FILE}")

Extracted 1000/50000 games
Extracted 2000/50000 games
Extracted 3000/50000 games
Extracted 4000/50000 games
Extracted 5000/50000 games
Extracted 6000/50000 games
Extracted 7000/50000 games
Extracted 8000/50000 games
Extracted 9000/50000 games
Extracted 10000/50000 games
Extracted 11000/50000 games
Extracted 12000/50000 games
Extracted 13000/50000 games
Extracted 14000/50000 games
Extracted 15000/50000 games
Extracted 16000/50000 games
Extracted 17000/50000 games
Extracted 18000/50000 games
Extracted 19000/50000 games
Extracted 20000/50000 games
Extracted 21000/50000 games
Extracted 22000/50000 games
Extracted 23000/50000 games
Extracted 24000/50000 games
Extracted 25000/50000 games
Extracted 26000/50000 games
Extracted 27000/50000 games
Extracted 28000/50000 games
Extracted 29000/50000 games
Extracted 30000/50000 games
Extracted 31000/50000 games
Extracted 32000/50000 games
Extracted 33000/50000 games
Extracted 34000/50000 games
Extracted 35000/50000 games
Extracted 36000/50000 games
E

In [125]:
%%writefile train.py

import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import math

games = []
unique_uci_moves = set()

with open("games.txt", "r", encoding="utf-8") as f:
    for line in f:
        moves = line.strip().split()

        if moves:
            games.append(moves)
            unique_uci_moves.update(moves)

print("Games:", len(games))
print("Unique UCI moves:", len(unique_uci_moves))

# full move uci tokenizer
vocab = ['.'] + sorted(set(unique_uci_moves))
vocab_size = len(sorted(vocab)) # all the unique moves are the vocabulary
move_to_id = {move:i for i, move in enumerate(vocab)}
id_to_move = {i:move for i, move in enumerate(vocab)}
encode = lambda moves: [move_to_id[move] for move in moves]
decode = lambda ids: ["".join(id_to_move[id] for id in ids)]

print(encode(["e2e4"]))
print(decode(encode(["e2e4"])))

import random
random.seed(42)
random.shuffle(games)

n1 = int(0.8 * len(games))
n2 = int(0.9 * len(games))

train_games = games[:n1]
val_games = games[n1:n2]
test_games = games[n2:]

def get_batch_cpu(games, batch_size, context_size, move_to_id):
    X = torch.zeros(
        (batch_size, context_size),
        dtype=torch.long
    )

    Y = torch.zeros(
        batch_size,
        dtype=torch.long
    )

    for i in range(batch_size):
        game = games[torch.randint(len(games), (1,)).item()]
        pos = torch.randint(len(game), (1,)).item()
        context = [0] * context_size

        for move in game[:pos]:
            ix = move_to_id[move]
            context = context[1:] + [ix]

        X[i] = torch.tensor(context)
        Y[i] = move_to_id[game[pos]]

    return X, Y

global_batch_size = 128
num_gpus = 2
batch_size = global_batch_size // num_gpus   # 64

eval_iters = 200
n_blocks = 12
n_embd = 224
n_head = 4
h_dim = 4 * n_embd
context_size = 128

eval_batches = {
    "train": [
        get_batch_cpu(train_games, batch_size, context_size, move_to_id)
        for _ in range(eval_iters)
    ],
    "val": [
        get_batch_cpu(val_games, batch_size, context_size, move_to_id)
        for _ in range(eval_iters)
    ],
}

local_rank = int(os.environ["LOCAL_RANK"])
generator = torch.Generator()
generator.manual_seed(1234 + local_rank)

class MLP(nn.Module):
  def __init__(self, n_embd, h_dim):
    super().__init__()
    self.hidden_layer = nn.Linear(n_embd, h_dim)
    self.relu = nn.ReLU()
    self.output_layer = nn.Linear(h_dim, n_embd)

  def forward(self, x):
    x = self.relu(self.hidden_layer(x))
    out = self.output_layer(x)
    return out


class Head(nn.Module):
  def __init__(self, n_embd, head_size, context_size):
    super().__init__()
    self.head_size = head_size
    self.Q = nn.Linear(n_embd, head_size)
    self.K = nn.Linear(n_embd, head_size)
    self.V = nn.Linear(n_embd, head_size)
    self.register_buffer(
        "tril",
        torch.tril(torch.ones(context_size, context_size)) # to match (B, T, T) each example will have T * T attention matrix where each row is a query and each column of that row is the key it needs to attend
    )

  def forward(self, x):
    q = self.Q(x)
    k = self.K(x)
    v = self.V(x)

    s = q @ k.transpose(-2, -1)
    s = s / math.sqrt(self.head_size)

    # causal mask
    mask = self.tril[:x.size(1), :x.size(1)]
    s = s.masked_fill(mask == 0, float("-inf"))

    w = F.softmax(s, dim=-1)

    o = w @ v

    return o


class MHA(nn.Module):
  def __init__(self, n_embd, head_size, n_head, context_size):
    super().__init__()
    self.heads = nn.ModuleList([
        Head(n_embd, head_size, context_size) for _ in range(n_head)
    ])
    self.out_layer = nn.Linear(n_embd, n_embd)

  def forward(self, x):
    outputs = [head(x) for head in self.heads]
    out = torch.cat(outputs, dim=-1)
    out = self.out_layer(out)
    return out


class Block(nn.Module):
  def __init__(self, n_embd, n_head, h_dim, context_size):
    super().__init__()
    head_size = n_embd // n_head
    self.mha = MHA(n_embd, head_size, n_head, context_size)
    self.mlp = MLP(n_embd, h_dim)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    x1 = x + self.mha(self.ln1(x))
    x2 = x1 + self.mlp(self.ln2(x1))
    return x2


class GPT(nn.Module):
  def __init__(self):
    super().__init__()
    self.blocks = nn.ModuleList([
        Block(n_embd, n_head, h_dim, context_size) for _ in range(n_blocks)
    ])
    self.embd_table = nn.Embedding(vocab_size, n_embd)
    self.pos_embd_table = nn.Embedding(context_size, n_embd)
    self.final_ln = nn.LayerNorm(n_embd)
    self.linear = nn.Linear(n_embd, vocab_size)

  def forward(self, x, y=None):
    tok_emb = self.embd_table(x)

    T = tok_emb.shape[1]
    pos_ids = torch.arange(T, device=x.device)
    pos_emb = self.pos_embd_table(pos_ids)

    x = tok_emb + pos_emb

    for block in self.blocks:
      x = block(x)

    x = self.final_ln(x)
    logits = self.linear(x)
    logits = logits[:, -1, :]
    if y is None:
      return logits
    loss = F.cross_entropy(logits, y)
    return logits, loss

# model
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

dist.init_process_group("nccl")

torch.cuda.set_device(local_rank)

device = torch.device(f"cuda:{local_rank}")

model = GPT()
model = model.to(device)

model = DDP(model, device_ids=[local_rank])

model = model.to(device)

print("Model device:", next(model.parameters()).device)

# optimizer
lr = 3e-4
wd = 0.01

total_params = sum(p.numel() for p in model.parameters())
print(total_params)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

# pytorch & cuda versions check
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

print("Device:", next(model.parameters()).device)

# measure peak memory before training
num_params = sum(p.numel() for p in model.parameters())
model_memory_mb = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
) / 1024**2

print(f"Model parameters: {num_params / 1e6:.2f}M")
print(f"Model parameter memory: {model_memory_mb:.1f} MB")

print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


# -------------------------------------------------------------------------------------------------
# TRAINING BELOW THIS
# -------------------------------------------------------------------------------------------------

# train
import time
import gc
import torch

# CUDA OOM Related ------------------------------
# Safely delete any older model and optimizer references to release the GPU memory
if 'model' in globals():
    del model
if 'optimizer' in globals():
    del optimizer

# 1. Clean up lingering variables and clear GPU cache to recover the 14+ GB of wasted VRAM
gc.collect()
torch.cuda.empty_cache()

# Now, recreate the model and optimizer fresh in clean memory
from __main__ import GPT, device  # Ensure imports are present
model = GPT().to(device)

# Training code ----------------------------------
lr = 3e-4
wd = 0.01
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

best_dev_loss = float("inf")
num_steps = 25000
log_step = num_steps // 10

torch.cuda.synchronize()
start = time.time()

for step in range(num_steps):

    # -------------------------
    # Get a random training batch
    # -------------------------
    x, y = get_batch_cpu(
        train_games,
        batch_size,
        context_size,
        move_to_id
    )

    x = x.to(device)
    y = y.to(device)

    # -------------------------
    # Forward pass
    # -------------------------
    logits, loss = model(x, y)
    if step % log_step == 0:
      print(loss.item())

    # -------------------------
    # Backward pass
    # -------------------------
    # set_to_none=True modestly reduces memory footprint during backward step
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

torch.cuda.synchronize()
elapsed = time.time() - start

# Throughput
total_samples = num_steps * batch_size
throughput = total_samples / elapsed

print(f"Training time: {elapsed / 60:.2f} min")
print(f"Time / step: {elapsed / num_steps * 1000:.2f} ms")
print(f"Throughput: {throughput:.1f} samples/sec")
print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

# measure peak memory after training
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print(f"Peak:      {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

# evaluate train and val los
@torch.no_grad()
def estimate_loss():
    model.eval()

    out = {}

    for split in ["train", "val"]:
        total_loss = 0.0

        for X, Y in eval_batches[split]:
            X = X.to(device)
            Y = Y.to(device)

            _, loss = model(X, Y)
            total_loss += loss.item()

        out[split] = total_loss / len(eval_batches[split])

    model.train()
    return out

estimate_loss()


# generate moves
def generate(max_new_tokens=50):
  context = [0] * context_size
  out = []
  for _ in range(max_new_tokens):
    # Create the tensor on the exact same device as the model (GPU)
    X = torch.tensor([context], device=device)
    logits = model(X)
    probs = F.softmax(logits, dim=-1)
    next_id = torch.multinomial(probs, num_samples=1)
    tok_id = next_id.item()
    out.append(tok_id)
    context = context[1:] + [tok_id]

    if tok_id == 0:
      break

  return out

generated_games = [decode(generate())[0] for _ in range(50)]


# test legality of moves
import chess
import numpy as np

positions = []

for moves in generated_games:
    moves = [moves[i:i+4] for i in range(0, len(moves), 4)]
    board = chess.Board()

    for i, move_uci in enumerate(moves):
        move = chess.Move.from_uci(move_uci)

        if move in board.legal_moves:
            board.push(move)
        else:
            positions.append(i + 1)
            break
    else:
        # Entire generated game was legal
        positions.append(len(moves) + 1)


positions = np.array(positions)

print("Mean first illegal position:", positions.mean())
print("Median:", np.median(positions))
print("Survive 10 moves:", np.mean(positions > 10))
print("Survive 20 moves:", np.mean(positions > 20))
print("Survive 50 moves:", np.mean(positions > 50))

Overwriting train.py


In [ ]:
!torchrun --standalone --nproc_per_node=2 train.py

W1008 12:04:36.284000 1216 torch/distributed/run.py:851] 
W1008 12:04:36.284000 1216 torch/distributed/run.py:851] *****************************************
W1008 12:04:36.284000 1216 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1008 12:04:36.284000 1216 torch/distributed/run.py:851] *****************************************
Games: 50000
Unique UCI moves: 1891
[990]
['e2e4']
Games: 50000
Unique UCI moves: 1891
[990]
['e2e4']
Model device: cuda:1
Model device: cuda:0
8138916
8138916
PyTorch: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8
PyTorch: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8
GPU count: 2
Device: cuda:0
GPU count: 2
Device: cuda:1
Model parameters: 8.14M
Model parameter memory: 31.0 MB
Model parameters: 8.14M
Model parameter memory: 31.0 MB
Allocated: 0.06 GB
Alloca

In [28]:
# -------
# Does the model reach the MI threshold / latent structure shown ?
# Start the experiment on mech interp.
# -------
# Write up